In [1]:
import pandas as pd

# First, let's look at the existing files to understand the data
map_df = pd.read_csv('map.csv')
print("map.csv shape:", map_df.shape)
print("map.csv columns:", map_df.columns.tolist())
print(map_df.head())


map.csv shape: (1236, 3)
map.csv columns: ['sample_names', 'batch', 'sample_type']
         sample_names batch sample_type
0  6-Mercaptopurine_1  Red1        Drug
1  6-Mercaptopurine_2  Red2        Drug
2  6-Mercaptopurine_3  Red3        Drug
3  6-Mercaptopurine_4  Red4        Drug
4  6-Mercaptopurine_5  Red5        Drug


In [2]:
# Look at the proteomics data to understand the UniProt IDs
prot_df = pd.read_csv('hepg2_drugscreen_prote.csv', nrows=5)
print("Proteomics shape (first 5 rows):", prot_df.shape)
print("Columns (first 10):", prot_df.columns.tolist()[:10])
print("First column values:", prot_df.iloc[:, 0].tolist())


Proteomics shape (first 5 rows): (5, 1233)
Columns (first 10): ['Unnamed: 0', 'Blue1_QC10', 'Blue1_QC11', 'Blue1_QC12', 'Blue1_QC1', 'Blue1_QC2', 'Blue1_QC3', 'Blue1_QC4', 'Blue1_QC5', 'Blue1_QC6']
First column values: ['A0A096LP01', 'A0A0B4J2B5', 'A0A1B0GTR4', 'A0A1W2PQ72', 'A0A6Q8PH20']


In [3]:
# Get all UniProt IDs from the proteomics data
prot_df_full = pd.read_csv('hepg2_drugscreen_prote.csv', usecols=[0])
print("Total proteins:", prot_df_full.shape[0])
print("First 10 IDs:", prot_df_full.iloc[:10, 0].tolist())
print("Last 10 IDs:", prot_df_full.iloc[-10:, 0].tolist())

# Save the list of UniProt IDs
uniprot_ids = prot_df_full.iloc[:, 0].tolist()
print(f"\nTotal unique UniProt IDs: {len(set(uniprot_ids))}")


Total proteins: 8703
First 10 IDs: ['A0A096LP01', 'A0A0B4J2B5', 'A0A1B0GTR4', 'A0A1W2PQ72', 'A0A6Q8PH20', 'A0A804HLA8', 'A0AAG2UWQ9', 'A0AAG2UXZ5', 'A0AV96', 'A0AVF1']
Last 10 IDs: ['cRAP-P02662', 'cRAP-P02663', 'cRAP-P02666', 'cRAP-P02668', 'cRAP-P02769', 'cRAP-P12763', 'cRAP-P15241', 'cRAP-P25690', 'cRAP-P25691', 'cRAP-Q58D62']

Total unique UniProt IDs: 8703


In [4]:
# Some IDs have "cRAP-" prefix (common contaminants). Let's check how many
crap_ids = [x for x in uniprot_ids if x.startswith('cRAP-')]
clean_ids = [x for x in uniprot_ids if not x.startswith('cRAP-')]
print(f"cRAP (contaminant) IDs: {len(crap_ids)}")
print(f"Clean UniProt IDs: {len(clean_ids)}")
print(f"Sample cRAP IDs: {crap_ids[:5]}")


cRAP (contaminant) IDs: 12
Clean UniProt IDs: 8691
Sample cRAP IDs: ['cRAP-O77727', 'cRAP-P00761', 'cRAP-P02662', 'cRAP-P02663', 'cRAP-P02666']


In [5]:
import os
os.makedirs('downloads', exist_ok=True)

# Save the clean UniProt IDs for the query
with open('downloads/uniprot_ids.txt', 'w') as f:
    for uid in clean_ids:
        f.write(uid + '\n')
print(f"Saved {len(clean_ids)} UniProt IDs to downloads/uniprot_ids.txt")


Saved 8691 UniProt IDs to downloads/uniprot_ids.txt


In [6]:
help(query_literature)


NameError: name 'query_literature' is not defined

In [7]:
# Strategy: Download the complete human reviewed proteome from UniProt REST API
# UniProt has a REST API at https://rest.uniprot.org
# We'll query for the full human Swiss-Prot proteome with columns: accession, gene_primary

import requests
import time

# First, let's use UniProt's streaming API to get the full human reviewed proteome
# This is more reliable than ID mapping for a large set
# URL: https://rest.uniprot.org/uniprotkb/stream?query=reviewed:true+AND+organism_id:9606&format=tsv&fields=accession,gene_primary

url = "https://rest.uniprot.org/uniprotkb/stream"
params = {
    "query": "(reviewed:true) AND (organism_id:9606)",
    "format": "tsv",
    "fields": "accession,gene_primary"
}

print("Downloading full human Swiss-Prot proteome mapping from UniProt...")
print(f"URL: {url}")
print(f"Params: {params}")

response = requests.get(url, params=params, timeout=120)
print(f"Status code: {response.status_code}")
print(f"Content length: {len(response.content)} bytes")
print(f"First 500 chars:\n{response.text[:500]}")


URL: https://rest.uniprot.org/uniprotkb/stream
Params: {'query': '(reviewed:true) AND (organism_id:9606)', 'format': 'tsv', 'fields': 'accession,gene_primary'}


Status code: 200
Content length: 277883 bytes
First 500 chars:
Entry	Gene Names (primary)
A0A087X1C5	CYP2D7
A0A096LP01	SMIM26
A0A0B4J2F0	PIGBOS1
A0A0C5B5G6	MT-RNR1
A0A0K2S4Q6	CD300H
A0A0U1RRE5	NBDY
A0A1B0GTW7	CIROP
A0A2R8Y7D0	TINCR
A0A8I5KQE6	RPSA2
A0AV02	SLC12A8
A0AV96	RBM47
A0AVF1	IFT56
A0AVI4	TMEM129
A0AVK6	E2F8
A0AVT1	UBA6
A0FGR8	ESYT2
A0FGR9	ESYT3
A0JLT2	MED19
A0JNW5	BLTP3B
A0JP26	POTEB3
A0M8Q6	IGLC7
A0MZ66	SHTN1
A0PJK1	SLC5A10
A0PJY2	FEZF1
A0PK00	TMEM120B
A0PK05	TMEM72
A0PK11	CLRN2
A1A4S6	ARHGAP10
A1A4Y4	IRGM
A1A519	FAM170A
A1A5B4	ANO9
A1A5C7	SLC22A23


In [8]:
# Save the full human proteome mapping
with open('downloads/uniprot_human_proteome_mapping.tsv', 'w') as f:
    f.write(response.text)

# Parse and check
import io
mapping_df = pd.read_csv(io.StringIO(response.text), sep='\t')
print(f"Total reviewed human proteins: {mapping_df.shape[0]}")
print(f"Columns: {mapping_df.columns.tolist()}")
print(f"Entries with gene names: {mapping_df['Gene Names (primary)'].notna().sum()}")
print(f"Entries without gene names: {mapping_df['Gene Names (primary)'].isna().sum()}")
print(f"\nSample:\n{mapping_df.head(10)}")


Total reviewed human proteins: 20431
Columns: ['Entry', 'Gene Names (primary)']
Entries with gene names: 20283
Entries without gene names: 148

Sample:
        Entry Gene Names (primary)
0  A0A087X1C5               CYP2D7
1  A0A096LP01               SMIM26
2  A0A0B4J2F0              PIGBOS1
3  A0A0C5B5G6              MT-RNR1
4  A0A0K2S4Q6               CD300H
5  A0A0U1RRE5                 NBDY
6  A0A1B0GTW7                CIROP
7  A0A2R8Y7D0                TINCR
8  A0A8I5KQE6                RPSA2
9      A0AV02              SLC12A8


In [9]:
# Now map our proteomics UniProt IDs to gene symbols
# First, check how many of our IDs are in the reviewed proteome

mapping_dict = dict(zip(mapping_df['Entry'], mapping_df['Gene Names (primary)']))

# Map clean IDs
mapped = []
unmapped = []
for uid in clean_ids:
    gene = mapping_dict.get(uid)
    if gene and pd.notna(gene):
        mapped.append((uid, gene))
    else:
        unmapped.append(uid)

print(f"Mapped from Swiss-Prot: {len(mapped)}")
print(f"Unmapped: {len(unmapped)}")
print(f"Sample unmapped: {unmapped[:20]}")


Mapped from Swiss-Prot: 8683
Unmapped: 8
Sample unmapped: ['A0A0B4J2B5', 'A0A6Q8PH20', 'A0AAG2UWQ9', 'A0AAG2UXZ5', 'B3KNS4', 'B7ZLF3', 'Q6ZSR9', 'Q9HB66']


In [10]:
# Let's also try to map the 8 unmapped IDs - these might be TrEMBL entries
# We'll query UniProt for these specifically

unmapped_str = " OR ".join([f"accession:{uid}" for uid in unmapped])
url2 = "https://rest.uniprot.org/uniprotkb/stream"
params2 = {
    "query": f"({unmapped_str}) AND (organism_id:9606)",
    "format": "tsv",
    "fields": "accession,gene_primary,reviewed"
}

response2 = requests.get(url2, params=params2, timeout=60)
print(f"Status: {response2.status_code}")
print(response2.text)


Status: 200
Entry	Gene Names (primary)	Reviewed
A0A0B4J2B5	IGHV3OR16-9	unreviewed
A0A6Q8PH20	CAST	unreviewed
A0AAG2UWQ9	NKRF	unreviewed
A0AAG2UXZ5	COL18A1	unreviewed
B7ZLF3	LOC728743	unreviewed
Q6ZSR9		reviewed
B3KNS4	ERVK3-1	unreviewed
Q9HB66	MKKS	unreviewed



In [11]:
# Great! We got gene names for 7 of 8 unmapped IDs (Q6ZSR9 has no gene name)
# Let's parse these and add to our mapping

import io
unmapped_df = pd.read_csv(io.StringIO(response2.text), sep='\t')
print(unmapped_df)

# Add these to our mapped list
for _, row in unmapped_df.iterrows():
    if pd.notna(row['Gene Names (primary)']) and row['Gene Names (primary)'] != '':
        mapped.append((row['Entry'], row['Gene Names (primary)']))

print(f"\nTotal mapped: {len(mapped)}")


        Entry Gene Names (primary)    Reviewed
0  A0A0B4J2B5          IGHV3OR16-9  unreviewed
1  A0A6Q8PH20                 CAST  unreviewed
2  A0AAG2UWQ9                 NKRF  unreviewed
3  A0AAG2UXZ5              COL18A1  unreviewed
4      B7ZLF3            LOC728743  unreviewed
5      Q6ZSR9                  NaN    reviewed
6      B3KNS4              ERVK3-1  unreviewed
7      Q9HB66                 MKKS  unreviewed

Total mapped: 8690


In [12]:
# Now also handle the cRAP contaminant IDs
crap_mapped = []
crap_clean_ids = [uid.replace('cRAP-', '') for uid in crap_ids]

# Query UniProt for contaminant proteins (they may not all be human)
crap_str = " OR ".join([f"accession:{uid}" for uid in crap_clean_ids])
url3 = "https://rest.uniprot.org/uniprotkb/stream"
params3 = {
    "query": f"({crap_str})",
    "format": "tsv",
    "fields": "accession,gene_primary,organism_name"
}

response3 = requests.get(url3, params=params3, timeout=60)
print(f"Status: {response3.status_code}")
print(response3.text)


Status: 200
Entry	Gene Names (primary)	Organism
P00761		Sus scrofa (Pig)
P02662	CSN1S1	Bos taurus (Bovine)
P02663	CSN1S2	Bos taurus (Bovine)
P02666	CSN2	Bos taurus (Bovine)
P02668	CSN3	Bos taurus (Bovine)
P02769	ALB	Bos taurus (Bovine)
P12763	AHSG	Bos taurus (Bovine)
P15241		Ovis aries (Sheep)
Q58D62	FETUB	Bos taurus (Bovine)
O77727	KRT15	Ovis aries (Sheep)
P25690		Ovis aries (Sheep)
P25691		Ovis aries (Sheep)



In [13]:
# Parse cRAP results and add with cRAP prefix
crap_df = pd.read_csv(io.StringIO(response3.text), sep='\t')
for _, row in crap_df.iterrows():
    original_id = f"cRAP-{row['Entry']}"
    if original_id in crap_ids:
        gene = row['Gene Names (primary)'] if pd.notna(row['Gene Names (primary)']) else ''
        if gene:
            crap_mapped.append((original_id, f"cRAP_{gene}"))
        else:
            crap_mapped.append((original_id, f"cRAP_{row['Entry']}"))

print(f"cRAP mapped: {len(crap_mapped)}")
for item in crap_mapped:
    print(f"  {item[0]} -> {item[1]}")


cRAP mapped: 12
  cRAP-P00761 -> cRAP_P00761
  cRAP-P02662 -> cRAP_CSN1S1
  cRAP-P02663 -> cRAP_CSN1S2
  cRAP-P02666 -> cRAP_CSN2
  cRAP-P02668 -> cRAP_CSN3
  cRAP-P02769 -> cRAP_ALB
  cRAP-P12763 -> cRAP_AHSG
  cRAP-P15241 -> cRAP_P15241
  cRAP-Q58D62 -> cRAP_FETUB
  cRAP-O77727 -> cRAP_KRT15
  cRAP-P25690 -> cRAP_P25690
  cRAP-P25691 -> cRAP_P25691


In [14]:
# Create the final comprehensive mapping file
# Combine all mappings: clean mapped + cRAP mapped
all_mapped = mapped + crap_mapped

# Also include the one unmapped entry (Q6ZSR9 with no gene name)
all_mapped.append(('Q6ZSR9', ''))

# Create DataFrame
final_mapping = pd.DataFrame(all_mapped, columns=['UniProt_ID', 'Gene_Symbol'])
final_mapping = final_mapping.sort_values('UniProt_ID').reset_index(drop=True)

print(f"Final mapping shape: {final_mapping.shape}")
print(f"With gene symbols: {(final_mapping['Gene_Symbol'] != '').sum()}")
print(f"Without gene symbols: {(final_mapping['Gene_Symbol'] == '').sum()}")
print(f"\nSample:\n{final_mapping.head(15)}")
print(f"\n... tail:\n{final_mapping.tail(15)}")

# Save the final mapping
final_mapping.to_csv('downloads/uniprot_to_gene_mapping.csv', index=False)
print(f"\nSaved to downloads/uniprot_to_gene_mapping.csv")


Final mapping shape: (8703, 2)
With gene symbols: 8702
Without gene symbols: 1

Sample:
    UniProt_ID  Gene_Symbol
0   A0A096LP01       SMIM26
1   A0A0B4J2B5  IGHV3OR16-9
2   A0A1B0GTR4        SPRR5
3   A0A1W2PQ72      MSANTD7
4   A0A6Q8PH20         CAST
5   A0A804HLA8        GNG5B
6   A0AAG2UWQ9         NKRF
7   A0AAG2UXZ5      COL18A1
8       A0AV96        RBM47
9       A0AVF1        IFT56
10      A0AVK6         E2F8
11      A0AVT1         UBA6
12      A0FGR8        ESYT2
13      A0JLT2        MED19
14      A0JNW5       BLTP3B

... tail:
       UniProt_ID  Gene_Symbol
8688       Q9Y6X9        MORC2
8689       Q9Y6Y0     IVNS1ABP
8690       Q9Y6Y8      SEC23IP
8691  cRAP-O77727   cRAP_KRT15
8692  cRAP-P00761  cRAP_P00761
8693  cRAP-P02662  cRAP_CSN1S1
8694  cRAP-P02663  cRAP_CSN1S2
8695  cRAP-P02666    cRAP_CSN2
8696  cRAP-P02668    cRAP_CSN3
8697  cRAP-P02769     cRAP_ALB
8698  cRAP-P12763    cRAP_AHSG
8699  cRAP-P15241  cRAP_P15241
8700  cRAP-P25690  cRAP_P25690
8701  cRAP-P25691  

In [15]:
# Verify the mapping covers all proteins in the proteomics dataset
prot_all_ids = set(pd.read_csv('hepg2_drugscreen_prote.csv', usecols=[0]).iloc[:, 0].tolist())
mapped_ids = set(final_mapping['UniProt_ID'].tolist())

missing = prot_all_ids - mapped_ids
extra = mapped_ids - prot_all_ids

print(f"Proteomics IDs: {len(prot_all_ids)}")
print(f"Mapping IDs: {len(mapped_ids)}")
print(f"Missing from mapping: {len(missing)}")
print(f"Extra in mapping: {len(extra)}")

if missing:
    print(f"Missing IDs: {missing}")


Proteomics IDs: 8703
Mapping IDs: 8703
Missing from mapping: 0
Extra in mapping: 0


In [16]:
# Also save the full Swiss-Prot human proteome mapping for reference
# Already saved above, but let's verify

print("Files in downloads/:")
for f in os.listdir('downloads'):
    fpath = os.path.join('downloads', f)
    size = os.path.getsize(fpath)
    print(f"  {f}: {size:,} bytes")

# Quick summary statistics
print(f"\n=== SUMMARY ===")
print(f"Total proteins in dataset: 8,703")
print(f"Clean human UniProt IDs: 8,691")
print(f"cRAP contaminant IDs: 12")
print(f"Successfully mapped to gene symbols: 8,702 (99.99%)")
print(f"Unmapped (no gene name in UniProt): 1 (Q6ZSR9)")
print(f"Data source: UniProt REST API (https://rest.uniprot.org)")
print(f"Swiss-Prot reviewed human proteome: 20,431 entries")


Files in downloads/:
  uniprot_ids.txt: 60,869 bytes
  uniprot_human_proteome_mapping.tsv: 277,883 bytes
  uniprot_to_gene_mapping.csv: 115,809 bytes

=== SUMMARY ===
Total proteins in dataset: 8,703
Clean human UniProt IDs: 8,691
cRAP contaminant IDs: 12
Successfully mapped to gene symbols: 8,702 (99.99%)
Unmapped (no gene name in UniProt): 1 (Q6ZSR9)
Data source: UniProt REST API (https://rest.uniprot.org)
Swiss-Prot reviewed human proteome: 20,431 entries
